# unialg Explorer

A gentle tour of the current `unialg` API.

A **morphism** is a typed arrow. You can run it, compose it, branch it, add parameters, add effects, and lift it through reusable data shapes.

The notebook keeps backend setup in `explore_support.py` so the main path can stay focused on the algebraic API.

## Setup

Run this once before the examples. It loads the local package and imports a small set of demo morphisms.

In [1]:
import sys
_VENV = "/home/scanbot/unialg/.venv/lib/python3.12/site-packages"
_SRC = "/home/scanbot/unialg/src"
for _p in (_VENV, _SRC):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from unialg import (
    identity, copy, delete, fst, snd, inl, inr,
    compose, par, pair, case,
    assoc, symmetry,
    Id, One, Const, Sum, Prod,
    Functor, Optic, Semiring,
    act, act_forward, act_backward, compose_optic,
    cata, ana, hylo,
    lower, apply_poly, poly_fmap
)
# from src.unialg.semantics.functors import apply_poly
# from src.unialg.structure.actions import poly_fmap
from explore_support import *

print("Ready.")

Ready.


## 1. Plain Morphisms

`add1` and `double` are ordinary morphisms from `Int` to `Int`. The helper `show_morphism` prints the logical type, and `run_int` runs a morphism on an integer input.

In [2]:
print("add1: ", show_morphism(add1))
print("double:", show_morphism(double))

print("add1(5)  =", run_int(add1, 5))
print("double(5) =", run_int(double, 5))

add1:  Int -> Int
double: Int -> Int
add1(5)  = 1
double(5) = 2


## 2. Composition

`compose(f, g)` means: run `f`, then run `g`. The output type of `f` must match the input type of `g`.

In [3]:
add_then_double = compose(add1, double)

print("add_then_double:", show_morphism(add_then_double))
print("add_then_double(3) =", run_int(add_then_double, 3))

add_then_double: Int -> Int
add_then_double(3) = 3


In [4]:
try:
    compose(add1, add_pair)
except Exception as e:
    print(type(e).__name__ + ": add1 returns Int, but add_pair expects Int x Int")

MorphismError: add1 returns Int, but add_pair expects Int x Int


## 3. Products

Products let us talk about pairs.

`copy` duplicates one input. `fst` and `snd` project from a pair. `pair(f, g)` sends the same input to both morphisms. `par(f, g)` sends the left input to `f` and the right input to `g`.

In [5]:
copy_int = copy(INT)
fst_int = fst(INT_PAIR)
snd_int = snd(INT_PAIR)

print("copy:", show_morphism(copy_int), "   copy(7) =", run_pair(copy_int, 7))
print("fst: ", show_morphism(fst_int), "   fst(3, 9) =", run_int_from_pair(fst_int, (3, 9)))
print("snd: ", show_morphism(snd_int), "   snd(3, 9) =", run_int_from_pair(snd_int, (3, 9)))

copy: Int -> Int x Int    copy(7) = (7, 7)
fst:  Int x Int -> Int    fst(3, 9) = 3
snd:  Int x Int -> Int    snd(3, 9) = 9


In [6]:
same_input = pair(add1, double)
side_by_side = par(add1, double)

print("pair(add1, double):", show_morphism(same_input))
print("pair(add1, double)(5) =", run_pair(same_input, 5))

print("par(add1, double): ", show_morphism(side_by_side))
print("par(add1, double)(5, 6) =", run_pair(side_by_side, (5, 6)))

pair(add1, double): Int -> Int x Int
pair(add1, double)(5) = (5, 5)
par(add1, double):  Int x Int -> Int x Int


AttributeError: 'tuple' object has no attribute 'value'

## 4. Sums

Sums represent a choice: a value arrives from the left side or the right side.

`case(f, g)` branches on that choice. Left values go to `f`; right values go to `g`.

In [ ]:
branch = case(add1, double)

print("branch:", show_morphism(branch))
print("branch(Left 5)  =", run_left_int(branch, 5))
print("branch(Right 5) =", run_right_int(branch, 5))

## 5. Parameters

A parametric morphism has an extra environment value. In this example, the parameter is an integer that gets added to the input.

The important surface idea is simple: provide a parameter and an input. The support helper owns the runtime packing.

In [ ]:
print("add_param:", show_morphism(add_param))
print("add_param(param=10, value=3) =", run_para_int(add_param, param=10, value=3))

In [ ]:
two_params = compose(add_param, add_param_again)

print("compose(add_param, add_param_again):", show_morphism(two_params))
print("with f_param=10, g_param=20, value=3:",
      run_composed_para_int(two_params, f_param=10, g_param=20, value=3))

## 6. Effects

A lax morphism returns its result inside an effect.

Here `Maybe` means the result is wrapped as an optional value, and `List` means the morphism can return multiple results. Composition sequences those effects automatically.

In [ ]:
maybe_pipeline = compose(maybe_add1, maybe_double)

print("maybe_pipeline:", show_morphism(maybe_pipeline))
print("maybe_pipeline(3) =", run_maybe_int(maybe_pipeline, 3))

In [ ]:
list_pipeline = compose(list_step, list_double)

print("list_pipeline:", show_morphism(list_pipeline))
print("list_pipeline(3) =", run_list_int(list_pipeline, 3))

## 7. Shapes / Polynomial Functors

A polynomial functor is a reusable data shape. `Id()` marks the places where values of the current type live.

`poly_fmap(shape, h)` lifts a morphism through the shape, applying `h` at every `Id()` position.

In [ ]:
pair_shape = Prod(Id(), Id())

print("shape:", pair_shape.pretty())
print("shape applied to Int:", show_type(apply_poly(pair_shape, INT)))

In [ ]:
lifted = poly_fmap(Functor("Test", pair_shape), add1)

print("lifted add1:", show_morphism(lifted))
print("lifted add1 on (3, 7) =", run_pair(lifted, (3, 7)))

In [ ]:
lifted_param = poly_fmap(Functor("Test", pair_shape), add_param)

print("lifted add_param:", show_morphism(lifted_param))
print("param=10, value=(3, 7):", run_para_pair(lifted_param, param=10, value=(3, 7)))

In [ ]:
lifted_maybe = poly_fmap(Functor("Test", pair_shape), maybe_add1)

print("lifted maybe_add1:", show_morphism(lifted_maybe))
print("lifted maybe_add1 on (3, 7) =", run_maybe_pair(lifted_maybe, (3, 7)))

In [ ]:
sum_shape = Sum(Id(), Id())
lifted_sum = poly_fmap(Functor("Test", sum_shape), add1)

print("sum shape:", sum_shape.pretty())
print("Left 5  ->", run_sum_int(lifted_sum, side="left", value=5))
print("Right 5 ->", run_sum_int(lifted_sum, side="right", value=5))

## 8. Structural Morphisms — assoc and symmetry

`assoc` and `symmetry` are canonical structural morphisms built from types alone — no primitives needed.

`assoc((A×B)×C)` gives the reassociation `(A×B)×C → A×(B×C)`.  
`symmetry(A×B)` gives the swap `A×B → B×A`.

In [ ]:
assoc_m = assoc(INT_TRIPLE_L)
print("assoc:", show_morphism(assoc_m))
print("assoc((1,2),3) =", run_assoc(assoc_m, 1, 2, 3))

sym_m = symmetry(INT_PAIR)
print("symmetry:", show_morphism(sym_m))
print("symmetry(3,7) =", run_pair(sym_m, (3, 7)))

## 8. Lowering

The examples above use friendly runners. Underneath, `lower(morphism)` compiles the morphism expression into a raw Hydra term. `run(morphism, argument, ctx, graph)` applies that term and reduces it.

You usually do not need to look at the lowered term while using the API, but it is useful when checking the compiler boundary.

In [ ]:
compiled = lower(add_then_double)

print("expression:", add_then_double.node.pretty())
print("lowered term class:", type(compiled).__name__)

## 9. What Can Be Composed?

`compose(f, g)` accepts `Morphism` objects. It does not accept raw Python functions directly.

A Python function can still participate, but it first has to become a Hydra `Primitive`, then a `Morphism` can point at that primitive. That backend pattern belongs in fixture/support code, not in the main tutorial path.

In [ ]:
print("compose takes Morphism values:")
print("  add1 is", type(add1).__name__)
print("  lambda x: x + 1 is", type(lambda x: x + 1).__name__)

## 10. Appendix: Where Did The Fixtures Come From?

The sample morphisms and notebook-friendly runners live in `explore_support.py`.

That file contains the raw Hydra details: primitive names, `expr.Prim(...)` leaves, argument packing, result unwrapping, and the `ctx` / `graph` used by `run`. Keeping that code out of the main path makes this notebook about the `unialg` API rather than Hydra plumbing.

## 11. Try It Yourself

Some small experiments:

- Compose `double` then `add3` and run it on `5`.
- Build `pair(add1, add3)` and run it on `10`.
- Build `par(double, add3)` and run it on `(2, 8)`.
- Lift `double` through `pair_shape` with `poly_fmap`.
- Compose a plain morphism with a Maybe morphism and check the result.

In [ ]:
double_then_add3 = compose(double, add3)

print("double_then_add3:", show_morphism(double_then_add3))
print("double_then_add3(5) =", run_int(double_then_add3, 5))

In [ ]:
add3_then_double = compose(add3, double)
thendoubleagain = pair(add3_then_double, double)
print(run_pair(thendoubleagain, 21))

## 12. Optics

An `Optic` is a triple `(functor, forward, backward)` where the functor describes the container shape and the two morphisms decompose and reconstruct around it.

`act(optic, h)` applies `h` through the optic: decompose with `forward`, lift `h` through the functor with `poly_fmap`, then reconstruct with `backward`.

Here a **pair lens** focuses on the first element of `Int × Int` using the functor `F(X) = X × Int`.

In [ ]:
pair_lens = Optic(
    functor=Functor("PairFirst", Prod(Id(), Const(INT))),
    forward=identity(INT_PAIR),
    backward=identity(INT_PAIR),
)

print("pair_lens.source:", show_type(pair_lens.source))
print("pair_lens.target:", show_type(pair_lens.target))
print("pair_lens.focus: ", show_type(pair_lens.focus))

lifted = act(pair_lens, add1)
print("act(pair_lens, add1):", show_morphism(lifted))
print("act(pair_lens, add1)(3, 7) =", run_pair(lifted, (3, 7)))

## 13. Semiring

A `Semiring` captures the algebraic structure of two binary operations on a carrier type. All four components (`plus`, `times`, `zero`, `one`) are `Morphism` objects, so they participate in the typed composition machinery.

In [ ]:
int_semiring = Semiring(
    name="int-add-mul",
    carrier=INT,
    plus=add_pair,
    times=mul_pair,
    zero=const_zero,
    one=const_one,
)

print("name:    ", int_semiring.name)
print("carrier: ", show_type(int_semiring.carrier))
print("plus:    ", show_morphism(int_semiring.plus))
print("times:   ", show_morphism(int_semiring.times))
print("zero:    ", show_morphism(int_semiring.zero))
print("one:     ", show_morphism(int_semiring.one))

print("plus(3, 4) =", run_int_from_pair(int_semiring.plus, (3, 4)))
print("times(3,4) =", run_int_from_pair(int_semiring.times, (3, 4)))

In [ ]:
def check_commutative(op, name):
    # op ∘ symmetry == op
    swapped = compose(symmetry(INT_PAIR), op)

    for a, b in [(2, 3), (0, 5), (-2, 7)]:
        lhs = run_int_from_pair(op, (a, b))
        rhs = run_int_from_pair(swapped, (a, b))
        print(f"{name} comm ({a}, {b}):", lhs, "==", rhs, lhs == rhs)


def check_associative(op, name):
    # left:  ((A × A) × A) --(op × id)--> A × A --op--> A
    left = compose(par(op, identity(INT)), op)

    # right: ((A × A) × A) --assoc--> A × (A × A) --(id × op)--> A × A --op--> A
    right = compose(
        compose(assoc(INT_TRIPLE_L), par(identity(INT), op)),
        op,
    )

    for a, b, c in [(1, 2, 3), (0, 4, 5), (-2, 7, 3)]:
        arg = nested_left_arg(a, b, c)
        lhs = int_val(run(left, arg, ctx, graph))
        rhs = int_val(run(right, arg, ctx, graph))
        print(f"{name} assoc ({a}, {b}, {c}):", lhs, "==", rhs, lhs == rhs)

In [ ]:
check_commutative(int_semiring.plus, "plus")
check_associative(int_semiring.plus, "plus")

check_commutative(int_semiring.times, "times")
check_associative(int_semiring.times, "times")

## 14. Recursive Schemes — cata, ana, hylo

Catamorphisms (`cata`), anamorphisms (`ana`), and hylomorphisms (`hylo`) are uniform recursion patterns over an `Optic` fixed point.

An `Optic` becomes a recursive carrier by supplying `forward = unroll` and `backward = roll` together with an explicit `carrier` type.

The demo below uses a trivially terminating carrier over `Int` with shape `F(X) = 1 + X`:
- `cata` folds: the algebra `1 + Int → Int` always returns 7
- `ana` unfolds: the coalgebra `Int → 1 + Int` always stops immediately, rolling to 42
- `hylo` chains them

In [ ]:
fp = one_or_self_optic(rolled_value=42)

stop_coalg = compose(delete(INT), inl(SumType(UNIT, INT)))
fold_alg   = case(const_int(7), identity(INT))

folded    = cata(fp, fold_alg)
unfolded  = ana(fp, stop_coalg)
transform = hylo(fp, stop_coalg, fold_alg)

print("cata: algebra folds any input to 7")
print("  cata(999) =", run_int(folded, 999))

print("ana:  coalgebra stops immediately, rolls to 42")
print("  ana(5) =", run_int(unfolded, 5))

print("hylo: unfold then fold → always 7")
print("  hylo(999) =", run_int(transform, 999))

## 15. Backend Morphisms

`BackendOps.from_spec` loads a JSON spec and registers each operation as a Hydra primitive with a canonical name (`unialg.backend.<op>`).  The same name is used regardless of which library backs it — swapping backends means changing the spec file, not the DSL.

`run` automatically includes `aux_primitives` from the morphism, so no manual graph extension is needed.

In [7]:
from pathlib import Path
import math as _math

from unialg import run
from unialg.emitters.backend import BackendOps
import hydra.dsl.meta.phantoms as P

_SPEC = Path("/home/scanbot/unialg/src/unialg/emitters/backends/jax.json")
ops = BackendOps.from_spec(_SPEC)

print("Loaded ops:", ", ".join(sorted(ops.keys())))
print()


def _fval(term):
    return term.value.value.value


def run_f2(m, a, b):
    """Run a binary float morphism on (a, b)."""
    return _fval(run(m, P.pair(P.float64(a), P.float64(b)).value, ctx, graph))


def run_f1(m, a):
    """Run a unary float morphism on a."""
    return _fval(run(m, P.float64(a).value, ctx, graph))


# --- elementwise binary ops ---
print("add(2, 3)          =", run_f2(ops["add"],       2.0,  3.0))
print("subtract(5, 1.5)   =", run_f2(ops["subtract"],  5.0,  1.5))
print("multiply(2, 3)     =", run_f2(ops["multiply"],  2.0,  3.0))
print("divide(9, 2)       =", run_f2(ops["divide"],    9.0,  2.0))
print("power(2, 10)       =", run_f2(ops["power"],     2.0, 10.0))
print("maximum(3, 7)      =", run_f2(ops["maximum"],   3.0,  7.0))
print("minimum(3, 7)      =", run_f2(ops["minimum"],   3.0,  7.0))
print("logaddexp(0, 0)    =", run_f2(ops["logaddexp"], 0.0,  0.0))
print()

# --- unary ops ---
print("exp(1)             =", run_f1(ops["exp"],       1.0))
print("log(e)             =", run_f1(ops["log"],   _math.e))
print("log1p(0)           =", run_f1(ops["log1p"],     0.0))
print("sqrt(16)           =", run_f1(ops["sqrt"],     16.0))
print("square(5)          =", run_f1(ops["square"],    5.0))
print("abs(-3.5)          =", run_f1(ops["abs"],      -3.5))
print("neg(4)             =", run_f1(ops["neg"],       4.0))
print("tanh(0)            =", run_f1(ops["tanh"],      0.0))
print("reciprocal(4)      =", run_f1(ops["reciprocal"],4.0))
print("sigmoid(0)         =", run_f1(ops["sigmoid"],   0.0))

Loaded ops: abs, add, cos, divide, exp, log, log1p, logaddexp, maximum, minimum, multiply, neg, power, reciprocal, reduce.add, reduce.logaddexp, reduce.maximum, reduce.minimum, reduce.multiply, relu, sigmoid, sign, sin, softmax, softplus, sqrt, square, subtract, tanh



AttributeError: 'tuple' object has no attribute 'value'

## 16. Semiring Factory

Build typed semirings from backend op names. The factory resolves both the elementwise op and its reduction form automatically — you only name the base op.

In [8]:
from pathlib import Path
from unialg.emitters.backend import BackendOps
from unialg.emitters.semiring_factory import semiring_from_backend, register_semiring_op

# Load the numpy backend
spec = Path('src/unialg/emitters/backends/numpy.json')
ops = BackendOps.from_spec(spec)

# See what ops are available
print('Available ops:')
for name in sorted(ops.keys()):
    print(' ', name)

Available ops:
  abs
  add
  cos
  divide
  exp
  log
  log1p
  logaddexp
  maximum
  minimum
  multiply
  neg
  power
  reciprocal
  reduce.add
  reduce.logaddexp
  reduce.maximum
  reduce.minimum
  reduce.multiply
  sigmoid
  sign
  sin
  softmax
  softplus
  sqrt
  square
  subtract
  tanh


In [9]:
# Constant morphisms for zero and one — placeholders until tensor type is settled
zero_m = ops['add']      # stand-in: will be a proper 1 -> carrier morphism
one_m  = ops['multiply']

# Real semiring: plus=add, times=multiply, adjoint=divide
real = semiring_from_backend('real', 'add', 'multiply', zero_m, one_m, ops,
                             adjoint_op='divide')
print('real semiring:')
print('  plus        :', real.plus)
print('  times       :', real.times)
print('  plus_reduce :', real.plus_reduce)
print('  times_reduce:', real.times_reduce)
print('  adjoint     :', real.adjoint)
print('  adjoint_red :', real.adjoint_reduce)  # None — divide has no reduce form

real semiring:
  plus        : Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypePair(value=PairType(first=TypeLiteral(value=Name(value='float')), second=TypeLiteral(value=Name(value='float')))), cod=TypeLiteral(value=Name(value='float'))), param=<hydra.core.TypeUnit object at 0x7f28f4412840>, monad=None, aux_primitives=(Primitive(name=Name(value='unialg.backend.add'), type_scheme=TypeScheme(variables=(), body=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeLiteral(value=Name(value='float')))))), constraints=None), implementation=<function register_backend_primitive.<locals>.impl at 0x7f28dfdf02c0>),))
  times       : Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypePair(val

In [10]:
# Tropical semiring: plus=minimum, times=add
tropical = semiring_from_backend('tropical', 'minimum', 'add', zero_m, one_m, ops)

# Fuzzy / max-min semiring
fuzzy = semiring_from_backend('fuzzy', 'maximum', 'minimum', zero_m, one_m, ops)

print('tropical plus :', tropical.plus)
print('tropical times:', tropical.times)
print()
print('fuzzy plus    :', fuzzy.plus)
print('fuzzy times   :', fuzzy.times)

tropical plus : Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypePair(value=PairType(first=TypeLiteral(value=Name(value='float')), second=TypeLiteral(value=Name(value='float')))), cod=TypeLiteral(value=Name(value='float'))), param=<hydra.core.TypeUnit object at 0x7f28f4412840>, monad=None, aux_primitives=(Primitive(name=Name(value='unialg.backend.minimum'), type_scheme=TypeScheme(variables=(), body=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeLiteral(value=Name(value='float')))))), constraints=None), implementation=<function register_backend_primitive.<locals>.impl at 0x7f28dfdf11c0>),))
tropical times: Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypePair(value=PairType

In [11]:
# op_env() selects the operation pair for a contraction.
# adjoint=False → standard mode: ⊕_j A[i,j] ⊗ B[j,k]
# adjoint=True  → adjoint mode:  ⊗_j A[i,j] ⊘ B[j,k]

env = real.op_env(adjoint=False)
print('standard op_env:')
print('  product (⊗):', env['product'])   # multiply
print('  fold    (⊕):', env['fold'])      # reduce.add
print('  seed (zero):', env['seed'])      # zero morphism

# adjoint mode needs times_reduce to be present
real_with_reduce = semiring_from_backend('real', 'add', 'multiply', zero_m, one_m, ops,
                                          adjoint_op='divide')
try:
    env_adj = real_with_reduce.op_env(adjoint=True)
    print()
    print('adjoint op_env:')
    print('  product (⊘):', env_adj['product'])  # divide
    print('  fold    (⊗):', env_adj['fold'])     # reduce.multiply
    print('  seed  (one):', env_adj['seed'])
except ValueError as e:
    print('adjoint error:', e)

standard op_env:
  product (⊗): Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypePair(value=PairType(first=TypeLiteral(value=Name(value='float')), second=TypeLiteral(value=Name(value='float')))), cod=TypeLiteral(value=Name(value='float'))), param=<hydra.core.TypeUnit object at 0x7f28f4412840>, monad=None, aux_primitives=(Primitive(name=Name(value='unialg.backend.multiply'), type_scheme=TypeScheme(variables=(), body=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeLiteral(value=Name(value='float')))))), constraints=None), implementation=<function register_backend_primitive.<locals>.impl at 0x7f28dfdf0ae0>),))
  fold    (⊕): Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypeLit

### Custom / learned semiring ops

Register any callable as a semiring op. Provide `reduce_fn` to make it usable as a fold in contractions.

In [12]:
import numpy as np

# Register a custom plus op that doesn't exist in the built-in spec
ops2 = BackendOps.from_spec(spec)  # fresh instance — won't affect ops

def smooth_plus_binary(a, b):
    """Smooth max: log(exp(a) + exp(b))."""
    return np.logaddexp(a, b)

def smooth_plus_reduce(x):
    """Fold smooth_plus_binary over axis 0."""
    return np.logaddexp.reduce(x, axis=0)

register_semiring_op('smooth_plus', smooth_plus_binary, ops2, reduce_fn=smooth_plus_reduce)

print('smooth_plus registered    :', 'smooth_plus' in ops2)
print('reduce.smooth_plus exists :', 'reduce.smooth_plus' in ops2)
print('original ops untouched    :', 'smooth_plus' not in ops)

# Build a smooth-max semiring using the custom op
smooth_sr = semiring_from_backend('smooth_max', 'smooth_plus', 'multiply', zero_m, one_m, ops2)
print()
print('smooth_max semiring:')
print('  plus        :', smooth_sr.plus)
print('  plus_reduce :', smooth_sr.plus_reduce)


smooth_plus registered    : True
reduce.smooth_plus exists : True
original ops untouched    : True

smooth_max semiring:
  plus        : Morphism(node=Prim(raw=TermLambda(value=Lambda(parameter=Name(value='x'), domain=Nothing, body=TermVariable(value=Name(value='x')))), dom=TypePair(value=PairType(first=TypeLiteral(value=Name(value='float')), second=TypeLiteral(value=Name(value='float')))), cod=TypeLiteral(value=Name(value='float'))), param=<hydra.core.TypeUnit object at 0x7f28f4412840>, monad=None, aux_primitives=(Primitive(name=Name(value='unialg.backend.smooth_plus'), type_scheme=TypeScheme(variables=(), body=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeFunction(value=FunctionType(domain=TypeLiteral(value=Name(value='float')), codomain=TypeLiteral(value=Name(value='float')))))), constraints=None), implementation=<function register_backend_primitive.<locals>.impl at 0x7f2822254180>),))
  plus_reduce : Morphism(node=Prim(raw=TermLambda(va